# Understanding Area and Volume Estimation with `skimage.measure.regionprops`

Welcome to this interactive bioimage analysis tutorial! 

When we measure structures in biology (like cell area or nuclear volume), we are working with digitized representations of continuous objects. In `scikit-image`, the `regionprops` function calculates the **`area`** property by essentially counting the number of pixels (in 2D) or voxels (in 3D) belonging to a labeled region. 

In this notebook, we will explore two critical concepts:
1. **Digitization and Angles (2D):** How rasterization (representing shapes on a square grid) and rotation alter the pixel count of squares and disks.
2. **Spacing and Volume (3D):** How anisotropic voxel spacing (common in Z-stacks from confocal microscopes) scales our voxel count into physical volume. Note that in `scikit-image`, the property for $N$-dimensional volume is still called `area`.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.measure import label, regionprops, find_contours
from skimage.transform import rotate
from skimage.draw import disk
from skimage.morphology import square, disk, ball, footprint_rectangle
import ipywidgets as widgets
from IPython.display import display
import ipywidgets as widgets

## 1. The 2D World: Rasterization, Angles, and Contour Area

In continuous math, rotating a shape does not change its area. However, images are discrete grids of pixels. When we rotate a binary mask, interpolation and re-thresholding cause boundary pixels to fluctuate.

In this cell, we use `skimage.morphology` to generate a discrete `square` and `disk`. We will also compare two different ways to estimate area:
1. **Pixel Counting (`regionprops`):** Strictly counts the discrete pixels inside the mask.
2. **Marching Squares Polygon (`find_contours`):** Finds the sub-pixel boundary between the object and the background, allowing us to calculate the area of the enclosed geometric polygon.

Try adjusting the **Size** and the **Angle**. Notice how the Pixel Count jumps in discrete steps as the grid rasterizes the rotation, while the Marching Squares estimation tends to handle the sub-pixel boundaries a bit differently!

In [2]:
def polygon_area(contour):
    """Calculates the geometric area of a 2D polygon using the Shoelace formula."""
    r = contour[:, 0]
    c = contour[:, 1]
    return 0.5 * np.abs(np.dot(c, np.roll(r, 1)) - np.dot(r, np.roll(c, 1)))

def analyze_2d_area_advanced(size, angle):
    # 1. Create base images using morphology
    sq_width = size * 2 + 1
    sq = footprint_rectangle((sq_width, sq_width))
    circ = disk(size)
    
    # Create a large enough canvas to avoid cropping during rotation
    canvas_size = int(size * 4 + 10)
    sq_canvas = np.zeros((canvas_size, canvas_size), dtype=np.uint8)
    circ_canvas = np.zeros((canvas_size, canvas_size), dtype=np.uint8)
    
    # Center shapes on canvas
    start_sq = (canvas_size - sq.shape[0]) // 2
    sq_canvas[start_sq:start_sq+sq.shape[0], start_sq:start_sq+sq.shape[1]] = sq
    
    start_circ = (canvas_size - circ.shape[0]) // 2
    circ_canvas[start_circ:start_circ+circ.shape[0], start_circ:start_circ+circ.shape[1]] = circ
    
    # 2. Rotate shapes (order=0 ensures nearest-neighbor to keep masks binary)
    sq_rot = rotate(sq_canvas, angle, order=0, resize=False) > 0.5
    circ_rot = rotate(circ_canvas, angle, order=0, resize=False) > 0.5
    
    # 3. Calculate Area via Regionprops (Pixel Counting)
    prop_sq = regionprops(label(sq_rot))[0]
    prop_circ = regionprops(label(circ_rot))[0]
    
    # 4. Calculate Area via Marching Squares (find_contours)
    # level=0.5 finds the boundary halfway between 0 (bg) and 1 (fg)
    contours_sq = find_contours(sq_rot, level=0.5)
    area_ms_sq = polygon_area(contours_sq[0]) if contours_sq else 0
    
    contours_circ = find_contours(circ_rot, level=0.5)
    area_ms_circ = polygon_area(contours_circ[0]) if contours_circ else 0
    
    # 5. Plotting
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Plot Square
    axes[0].imshow(sq_rot, cmap='gray')
    if contours_sq:
        axes[0].plot(contours_sq[0][:, 1], contours_sq[0][:, 0], linewidth=2, color='red', label='Marching Squares')
    axes[0].set_title(f"Square (Half-width: {size})\n"
                      f"Regionprops (Pixels): {prop_sq.area}\n"
                      f"Marching Squares Area: {area_ms_sq:.1f}\n"
                      f"Theoretical Area: {sq_width**2}")
    axes[0].axis('off')
    axes[0].legend(loc='lower right')
    
    # Plot Disk
    axes[1].imshow(circ_rot, cmap='gray')
    if contours_circ:
        axes[1].plot(contours_circ[0][:, 1], contours_circ[0][:, 0], linewidth=2, color='red', label='Marching Squares')
    # For a discrete morphology disk, the effective continuous diameter is roughly (2*radius + 1)
    theoretical_disk = np.pi * (size + 0.5)**2
    axes[1].set_title(f"Disk (Radius: {size})\n"
                      f"Regionprops (Pixels): {prop_circ.area}\n"
                      f"Marching Squares Area: {area_ms_circ:.1f}\n"
                      f"Theoretical Area: ~{theoretical_disk:.1f}")
    axes[1].axis('off')
    axes[1].legend(loc='lower right')
    
    plt.suptitle(f"Rotation Angle: {angle}° | Shape Size: {size}", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Create interactive slider
widgets.interact(analyze_2d_area_advanced, 
                 size=widgets.IntSlider(min=2, max=30, step=1, value=10, description='Size'),
                 angle=widgets.IntSlider(min=0, max=90, step=1, value=0, description='Angle'));

interactive(children=(IntSlider(value=10, description='Size', max=30, min=2), IntSlider(value=0, description='…

## 2. The 3D World: Volume and Voxel Spacing

In 3D microscopy, voxels are rarely perfect cubes. Often, the lateral resolution (X and Y) is much higher than the axial resolution (Z). This means a single voxel represents a physical rectangular prism, not a cube. 

By default, `regionprops` just returns the raw voxel count. To get true physical volume, we must pass the `spacing` parameter, which is a tuple representing `(Z_spacing, Y_spacing, X_spacing)`. `scikit-image` will automatically scale the voxel count by the volume of a single voxel:

$$Volume_{physical} = N_{voxels} \times (S_z \times S_y \times S_x)$$

*Note: In `skimage`, the N-dimensional hyper-volume property is still accessed via `.area`.*

Let's test this with a discrete ball (radius $= 10$) and a discrete cube (width $= 21$).

In [3]:
def analyze_3d_volume(z_spacing, y_spacing, x_spacing):
    # 1. Generate 3D shapes
    img_ball = ball(10)  # Generates a 21x21x21 array
    img_cube = footprint_rectangle((21, 21, 21))  # Generates a 21x21x21 array
    
    # 2. Label regions
    label_ball = label(img_ball)
    label_cube = label(img_cube)
    
    # 3. Calculate spacing and voxel volume
    spacing = (z_spacing, y_spacing, x_spacing)
    voxel_volume = z_spacing * y_spacing * x_spacing
    
    # 4. Measure properties using the spacing parameter
    props_ball = regionprops(label_ball, spacing=spacing)[0]
    props_cube = regionprops(label_cube, spacing=spacing)[0]
    
    # 5. Output Results
    print(f"{'='*40}")
    print(f"SPACING CALIBRATION: {spacing}")
    print(f"Volume of 1 Voxel: {voxel_volume:.3f} physical units³")
    print(f"{'='*40}\n")
    
    print("🟢 DISCRETE BALL (Radius = 10 voxels):")
    print(f"  Voxel Count (Raw): {np.sum(img_ball)}")
    print(f"  Estimated Physical Volume: {props_ball.area:.2f}")
    # Theoretical continuous volume scaled by the discrete grid approximation
    theoretical_ball = (4/3) * np.pi * (10**3) * voxel_volume
    print(f"  Theoretical Continuous Volume: {theoretical_ball:.2f}\n")
    
    print("🟩 DISCRETE CUBE (21x21x21 voxels):")
    print(f"  Voxel Count (Raw): {np.sum(img_cube)}")
    print(f"  Estimated Physical Volume: {props_cube.area:.2f}")
    theoretical_cube = (21**3) * voxel_volume
    print(f"  Theoretical Continuous Volume: {theoretical_cube:.2f}")

# Create interactive sliders for X, Y, and Z spacing
widgets.interact(analyze_3d_volume, 
                 z_spacing=widgets.FloatSlider(min=0.1, max=3.0, step=0.1, value=1.0, description='Z Spacing'),
                 y_spacing=widgets.FloatSlider(min=0.1, max=3.0, step=0.1, value=1.0, description='Y Spacing'),
                 x_spacing=widgets.FloatSlider(min=0.1, max=3.0, step=0.1, value=1.0, description='X Spacing'));

interactive(children=(FloatSlider(value=1.0, description='Z Spacing', max=3.0, min=0.1), FloatSlider(value=1.0…